# 02 · Home and work locations

Step by step for one month, then a check against the synthetic truth.
The rules are those of the paper (see `notebook_docs/methodology/HOME_LOCATION.md` and
`WORK_LOCATION.md`); every threshold is in the `home_location` / `work_location`
blocks of the settings.

1. **Prepare** raw dwells: drop short and non-area dwells, clip to the study area,
   keep devices seen at night and in working hours on at least two dates.
2. **Home**: among each device's five most frequent 7-character geohashes, rank by
   night hours and by Shabbat hours; the best weighted rank with at least one Shabbat hour wins.
3. **Work**: the most frequent geohash in working hours on workdays that is more than
   1 km from home, never visited on Shabbat or holidays, and seen on at least two days.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from remote_work_utils.geo import geohash_polygons
from remote_work_utils.home_work import detect_homes, detect_work, homes_by_stat_area
from remote_work_utils.pipeline import run_prepare
from remote_work_utils.settings import load_settings, resolve_path

settings = load_settings("../configs/demo.json")
month = "202110"
truth = pd.read_csv(resolve_path(settings, "synthetic_users"))

## 1. Prepare

In [ ]:
raw = pd.read_parquet(resolve_path(settings, "dwells_raw", month=month))
dwells = run_prepare(settings, month)
print(f"{len(raw):,} raw dwells -> {len(dwells):,} prepared dwells, {dwells['identifier'].nunique()} devices")
dwells.head(3)

## 2. Home

In [ ]:
from remote_work_utils.schema import normalize_dwells

dwells = normalize_dwells(dwells, settings)
home = detect_homes(dwells, settings)
home.homes.head()

In [ ]:
pd.Series(home.metrics).round(3)

## 3. Work

In [ ]:
users_work_home = detect_work(dwells, home.homes, settings)
users_work_home.head()

## Check against the synthetic truth
A detected location is correct when the true home / office lies in the detected
geohash or within the configured radius of it.

In [ ]:
def hit_rate(detected, lat_col, lon_col, geohash_col):
    merged = detected.merge(truth, on="identifier")
    cells = gpd.GeoSeries(list(geohash_polygons(merged[geohash_col], settings["geohash"]["lonlat_swapped"])), crs=4326)
    points = gpd.GeoSeries(gpd.points_from_xy(merged[lon_col], merged[lat_col]), crs=4326)
    crs = settings["region"]["projected_crs"]
    distance = points.to_crs(crs).distance(cells.to_crs(crs))
    return (distance <= settings["remote_work"]["location_radius_meters"]).mean(), len(merged)

home_hit, n_home = hit_rate(home.homes.rename(columns={"geohash": "geohash_home"}), "home_lat", "home_lon", "geohash_home")
office = users_work_home.merge(truth.loc[truth["has_office"], ["identifier"]], on="identifier")
work_hit, n_work = hit_rate(office, "office_lat", "office_lon", "geohash_work")
pd.DataFrame(
    {"devices": [n_home, n_work], "correct (%)": [100 * home_hit, 100 * work_hit]},
    index=["home (all devices)", "work (devices with an office)"],
).round(1)

Most devices without an office still get a "work" location: the work-friendly third
place where they spend working hours most often. From location and timing alone the method
cannot tell a regular cafe from an office; notebook 04 adds POI information for this.

In [ ]:
no_office = users_work_home.merge(truth.loc[~truth["has_office"], ["identifier"]], on="identifier")
print(f"{len(no_office)} of {(~truth['has_office']).sum()} devices without an office received a work location")

## Homes per statistical area vs population

In [ ]:
stat_areas = gpd.read_file(resolve_path(settings, "stat_areas"))
areas, correlation = homes_by_stat_area(home.homes, stat_areas, settings)
areas = areas.rename(columns={"identifier": "num_homes"})  # legacy output names the count "identifier"
population = settings["stat_areas"]["population_column"]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(areas[population], areas["num_homes"], color="#2b6cb0")
slope, intercept = np.polyfit(areas[population], areas["num_homes"], 1)
x = np.linspace(areas[population].min(), areas[population].max(), 10)
ax.plot(x, slope * x + intercept, color="#dd6b20", label=f"Linear fit (Spearman ρ = {correlation:.2f})")
ax.set_title(f"Detected homes vs population by statistical area, {month}")
ax.set_xlabel("Population (persons)")
ax.set_ylabel("Detected homes (devices)")
ax.legend()
plt.show()

> **Fig. 2.1.** Detected home locations per statistical area against area population.
> The number of devices grows with population, as expected from a representative panel.
> *Notes:* synthetic data, one month; Spearman rank correlation.